# 1. Import Libraries

In [11]:
import sys
print(sys.executable)

/Library/Developer/CommandLineTools/usr/bin/python3


In [14]:
!/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install jinja2 

Defaulting to user installation because normal site-packages is not writeable
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 2. Read Dataset

In [2]:
data = pd.read_csv("FINAL_pollutants_deaths_population.csv")
data.shape
pd.DataFrame(data.columns)

,0
0,Country
1,Cause
2,Year
3,Deaths
4,Deaths Upper
5,Deaths Lower
6,Region Name
7,Exposure Lower HAP
8,Exposure Mean HAP
9,Exposure Upper HAP


# 3. Feature Selection & Engineering
Select only the required features and ignore the confidnce intervals for now

In [3]:
df = data[["Cause", "Country", "Year", "Deaths", "Region Name", "Exposure Mean HAP", 
                       "Exposure Mean NO2", "Exposure Mean OZONE", "Exposure Mean PM25", 
                       "Population"]]


pd.DataFrame(df.columns)

,0
0,Cause
1,Country
2,Year
3,Deaths
4,Region Name
5,Exposure Mean HAP
6,Exposure Mean NO2
7,Exposure Mean OZONE
8,Exposure Mean PM25
9,Population


## 3.1 Compute mortality rate

In [4]:
df["Mortality Rate"] = (df["Deaths"] / df["Population"]) * 100000
pd.DataFrame(df.columns)

/var/folders/82/w51l4pcn5351zd0bz7gqwcrr0000gq/T/ipykernel_48295/3964810113.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Mortality Rate"] = (df["Deaths"] / df["Population"]) * 100000


,0
0,Cause
1,Country
2,Year
3,Deaths
4,Region Name
5,Exposure Mean HAP
6,Exposure Mean NO2
7,Exposure Mean OZONE
8,Exposure Mean PM25
9,Population


# 4. Compute Correlations

The **weighted Pearson correlation** measures the linear association between two variables when each observation carries a different importance (weight).  
In this work, each country–year is weighted by its population, so that the correlation reflects the relationship **as experienced by the average person** rather than the average country.

---

### Formula (step-by-step)

First define the **weighted mean** of a variable \(x\):

$$
\mu_x = \frac{\sum_i w_i x_i}{\sum_i w_i},
\qquad
\mu_y = \frac{\sum_i w_i y_i}{\sum_i w_i}.
$$

The **weighted variance** and **weighted covariance** are:

$$
\operatorname{var}_w(x) = \frac{\sum_i w_i (x_i - \mu_x)^2}{\sum_i w_i},
$$

$$
\operatorname{cov}_w(x,y) = \frac{\sum_i w_i (x_i - \mu_x)(y_i - \mu_y)}{\sum_i w_i}.
$$

thus, the **weighted Pearson correlation** is defined as:

$$
\rho_w(x,y) \;=\;
\frac{\operatorname{cov}_w(x,y)}
     {\sqrt{\operatorname{var}_w(x)\,\operatorname{var}_w(y)}}.
$$

### Fully substituted formula

Expanding everything into a single expression gives:

$$
\rho_w(x,y) \;=\; 
\frac{\sum_i w_i (x_i - \mu_x)(y_i - \mu_y)}
     {\sqrt{\;\sum_i w_i (x_i - \mu_x)^2 \;\;\sum_i w_i (y_i - \mu_y)^2}}
$$


This reduces to the ordinary Pearson correlation if all weights $(w_i)$ are equal.

In [5]:
# Weighted Pearson correlation
def calculate_weighted_correlation(data_x: np.ndarray, data_y: np.ndarray, weights: np.ndarray) -> float:
    """
    Calculate the weighted Pearson correlation coefficient between two arrays.
    
    Parameters:
    -----------
    data_x : np.ndarray
        First array of values
    data_y : np.ndarray
        Second array of values
    weights : np.ndarray
        Array of weights corresponding to each pair of observations
        
    Returns:
    --------
    float
        Weighted Pearson correlation coefficient between -1 and 1
        Returns NaN if insufficient valid data or zero variance
    """
    # Convert inputs to float arrays
    weights = np.asarray(weights, dtype=float)
    data_x = np.asarray(data_x, dtype=float)
    data_y = np.asarray(data_y, dtype=float)
    
    # Filter out missing values
    valid_mask = np.isfinite(data_x) & np.isfinite(data_y) & np.isfinite(weights)
    if valid_mask.sum() < 3:  # Need at least 3 points for meaningful correlation
        return np.nan
        
    weights = weights[valid_mask]
    data_x = data_x[valid_mask]
    data_y = data_y[valid_mask]
    
    # Check if weights are valid
    if (weights <= 0).all():
        return np.nan
        
    total_weight = weights.sum()
    if total_weight == 0:
        return np.nan
        
    # Calculate weighted means
    weighted_mean_x = (weights * data_x).sum() / total_weight
    weighted_mean_y = (weights * data_y).sum() / total_weight
    
    # Calculate weighted covariance and variances
    weighted_covariance = (weights * (data_x - weighted_mean_x) * (data_y - weighted_mean_y)).sum() / total_weight
    weighted_variance_x = (weights * (data_x - weighted_mean_x) ** 2).sum() / total_weight
    weighted_variance_y = (weights * (data_y - weighted_mean_y) ** 2).sum() / total_weight
    
    # Check for zero variance
    if weighted_variance_x <= 0 or weighted_variance_y <= 0:
        return np.nan
        
    # Calculate correlation coefficient
    correlation = weighted_covariance / np.sqrt(weighted_variance_x * weighted_variance_y)
    return correlation

In [6]:
exp_cols = ["Exposure Mean PM25", "Exposure Mean NO2", 
            "Exposure Mean OZONE", "Exposure Mean HAP"]

records = []
for pol in exp_cols:
    for cause, g in df.groupby("Cause", sort=False):
        sub = g[[pol, "Mortality Rate", "Population"]].dropna()
        if len(sub) < 10:
            continue
        r = calculate_weighted_correlation(
            sub[pol].to_numpy(),
            sub["Mortality Rate"].to_numpy(),
            sub["Population"].to_numpy()
        )
        records.append({
            "Cause": cause,
            "Pollutant": pol.replace("Exposure Mean ", ""),  # nicer name
            "r": r,
            "n": len(sub)
        })

cause_corr_weighted = pd.DataFrame.from_records(records)

In [7]:
cause_corr_weighted = cause_corr_weighted.query("r > 0").sort_values(["Pollutant","r"], ascending=[True, False])

# 5. Render Tables of Causes of Deaths Related to Pollutants

In [8]:
# 3) Build "binned list of diseases per pollutant" tables
def binned_tables_with_lists(cause_corr, r_min= 0.30):
    """
    Returns a dict: {pollutant: (table_df, display_df)}
      - table_df: each cell is a Python list of cause names
      - display_df: same table but cells are pretty strings for viewing
    """
    # bins & labels
    bins = [0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.01]
    labels = [
        "0.4 ≤ r < 0.5",
        "0.5 ≤ r < 0.6",
        "0.6 ≤ r < 0.7",
        "0.7 ≤ r < 0.8",
        "0.8 ≤ r < 0.9",
        "0.9 ≤ r < 1.0",
    ]

    cc = cause_corr.loc[(cause_corr["r"] > r_min) & cause_corr["r"].notna()].copy()
    cc["bin"] = pd.cut(cc["r"], bins=bins, labels=labels, right=False, include_lowest=True)

    out = {}
    for pol, g in cc.groupby("Pollutant", sort=True):
        # gather list of causes per bin (no padding here)
        col_lists = {lab: g.loc[g["bin"].eq(lab), "Cause"].sort_values().tolist() for lab in labels}

        # build a rectangular DataFrame by padding with "" (so display is aligned)
        max_len = max((len(v) for v in col_lists.values()), default=0)
        padded = {k: (v + [""] * (max_len - len(v))) for k, v in col_lists.items()}
        table_df = pd.DataFrame(padded)[labels]  # order columns

        # add totals row (counts)
        totals = [str(len(col_lists[lab])) for lab in labels]
        table_df.loc["Total"] = totals

        # a pretty display version with bullets and line breaks
        def format_cell(cell):
            return "• " + "\n• ".join([x for x in cell if x]) if isinstance(cell, list) else cell

        display_df = pd.DataFrame({lab: [format_cell(col_lists[lab])] for lab in labels})
        display_df.index = [f"{pol} — Causes by r-bin (r ≥ {r_min:.2f})"]
        # add a second row with counts
        display_df.loc["Counts"] = [len(col_lists[lab]) for lab in labels]

        out[pol] = (table_df, display_df)
    return out

tables = binned_tables_with_lists(cause_corr_weighted, r_min=0.40)

# 4) Render neatly in a notebook:
for pol, (table_df, display_df) in tables.items():
    # a) the full rectangular table (one cause per cell/row) + totals row
    print(f"\n{pol} — Binned Positive Correlations (lists of diseases)")
    display(table_df.style.set_properties(**{"white-space":"nowrap"}))

    # b) a compact one-row view where each cell contains a bulleted list (plus a counts row)
    #display(display_df.style.set_properties(**{"white-space":"pre-wrap"}))


HAP — Binned Positive Correlations (lists of diseases)


,0.4 ≤ r < 0.5,0.5 ≤ r < 0.6,0.6 ≤ r < 0.7,0.7 ≤ r < 0.8,0.8 ≤ r < 0.9,0.9 ≤ r < 1.0
0,Appendicitis,Acute hepatitis A,Diarrheal diseases,,,
1,Ascariasis,Acute hepatitis C,Drug-susceptible tuberculosis,,,
2,Chronic hepatitis B including cirrhosis,Acute hepatitis E,Maternal hemorrhage,,,
3,Congenital heart anomalies,Asthma,Neonatal encephalopathy due to birth asphyxia and trauma,,,
4,Digestive congenital anomalies,Drowning,Other neonatal disorders,,,
5,Ectopic pregnancy,Gastritis and duodenitis,Varicella and herpes zoster,,,
6,Encephalitis,Hemolytic disease and other neonatal jaundice,,,,
7,Foreign body in other body part,Hepatoblastoma,,,,
8,G6PD deficiency,Indirect maternal deaths,,,,
9,Gonococcal infection,Maternal abortion and miscarriage,,,,



NO2 — Binned Positive Correlations (lists of diseases)


,0.4 ≤ r < 0.5,0.5 ≤ r < 0.6,0.6 ≤ r < 0.7,0.7 ≤ r < 0.8,0.8 ≤ r < 0.9,0.9 ≤ r < 1.0
0,Anorexia nervosa,Acute myeloid leukemia,Colon and rectum cancer,,,
1,Atrial fibrillation and flutter,Alzheimer's disease and other dementias,Other non-Hodgkin lymphoma,,,
2,Coal workers pneumoconiosis,Aortic aneurysm,Pancreatic cancer,,,
3,"Endocrine, metabolic, blood, and immune disorders",Bladder cancer,Rheumatoid arthritis,,,
4,Gallbladder and biliary tract cancer,Brain and central nervous system cancer,Total cancers,,,
5,Ischemic heart disease,Breast cancer,"Tracheal, bronchus, and lung cancer",,,
6,Lower extremity peripheral arterial disease,Chronic lymphoid leukemia,,,,
7,Malignant skin melanoma,Chronic myeloid leukemia,,,,
8,Mesothelioma,Kidney cancer,,,,
9,Multiple sclerosis,Motor neuron disease,,,,



OZONE — Binned Positive Correlations (lists of diseases)


,0.4 ≤ r < 0.5,0.5 ≤ r < 0.6,0.6 ≤ r < 0.7,0.7 ≤ r < 0.8,0.8 ≤ r < 0.9,0.9 ≤ r < 1.0
0,Chronic obstructive pulmonary disease,,Rheumatic heart disease,,,
Total,1,0,1,0,0,0



PM25 — Binned Positive Correlations (lists of diseases)


,0.4 ≤ r < 0.5,0.5 ≤ r < 0.6,0.6 ≤ r < 0.7,0.7 ≤ r < 0.8,0.8 ≤ r < 0.9,0.9 ≤ r < 1.0
0,Acute hepatitis E,Acute hepatitis A,Encephalitis,Rheumatic heart disease,,
1,Appendicitis,Hemolytic disease and other neonatal jaundice,Venomous animal contact,,,
2,Asthma,Paratyphoid fever,,,,
3,Chronic hepatitis B including cirrhosis,Typhoid fever,,,,
4,Diarrheal diseases,,,,,
5,Multidrug-resistant tuberculosis without extensive drug resistance,,,,,
6,Neonatal preterm birth,,,,,
7,Other nutritional deficiencies,,,,,
8,Rabies,,,,,
9,Total burden related to hepatitis B,,,,,


# 6. See Impact of Pollutants on Mortality

In [9]:
significant_causes = cause_corr_weighted[(cause_corr_weighted['r'] >= 0.5)]['Cause'].unique()
print("Total Significant Causes related to Pollutants are ", len(significant_causes))

Total Significant Causes related to Pollutants are  55


In [10]:
# Store the original number of rows
original_rows = len(df)

# Filter the DataFrame to keep only rows where Cause is in unique_top_causes
filtered_df = df[df['Cause'].isin(significant_causes)]

# Calculate statistics
retained_rows = len(filtered_df)
dropped_rows = original_rows - retained_rows

# Print statistics
print(f"Original number of rows: {original_rows}")
print(f"Rows dropped: {dropped_rows}")
print(f"Rows retained: {retained_rows}")
print(f"Percentage of rows retained: {retained_rows/original_rows*100:.2f}%")

Original number of rows: 1460844
Rows dropped: 1113024
Rows retained: 347820
Percentage of rows retained: 23.81%


In [15]:
def plot_pollutant_deaths_plotly(df: pd.DataFrame, filtered_df: pd.DataFrame):
    """Interactive Plotly chart:
    - Bars = % of deaths related to pollutants (left Y axis, 0–100%)
    - Line = total deaths (right Y axis)
    - Optionally saves to HTML if `save_html` path is provided
    """
    # Aggregate
    yearly_deaths_total = df.groupby('Year', as_index=False)['Deaths'].sum().rename(columns={'Deaths': 'Deaths_total'})
    yearly_deaths_filtered = filtered_df.groupby('Year', as_index=False)['Deaths'].sum().rename(columns={'Deaths': 'Deaths_filtered'})
    yearly_comparison = pd.merge(yearly_deaths_total, yearly_deaths_filtered, on='Year', how='outer')

    # Compute percentage
    yearly_comparison['percentage'] = (yearly_comparison['Deaths_filtered'] / yearly_comparison['Deaths_total']) * 100
    yearly_comparison.replace([np.inf, -np.inf], np.nan, inplace=True)
    yearly_comparison['percentage'] = yearly_comparison['percentage'].fillna(0.0)

    # Format
    yearly_comparison['Deaths_total'] = yearly_comparison['Deaths_total'].round(0).astype('Int64')
    yearly_comparison['Deaths_filtered'] = yearly_comparison['Deaths_filtered'].round(0).astype('Int64')
    yearly_comparison['percentage'] = yearly_comparison['percentage'].round(2)
    yearly_comparison = yearly_comparison.sort_values('Year')

    # Build plot
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Bars (percentage)
    fig.add_bar(
        x=yearly_comparison['Year'],
        y=yearly_comparison['percentage'],
        text=[f"{v:.2f}%" for v in yearly_comparison['percentage']],
        textposition='outside',
        name='Total deaths% (Pollutants)',
        hovertemplate=(
            "<br><b>Year %{x}</b><br>"
            "Percentage: %{y:.2f}%<br>"
            "Absolute: %{customdata[0]:,}<br>"
        ),
        customdata=np.stack([
            yearly_comparison['Deaths_filtered'].fillna(0).astype(int),
            yearly_comparison['Deaths_total'].fillna(0).astype(int)
        ], axis=-1),
        offsetgroup=0,
        showlegend=True,
    )

    # Line (total deaths)
    fig.add_trace(
        go.Scatter(
            x=yearly_comparison['Year'],
            y=yearly_comparison['Deaths_total'],
            mode='lines+markers',
            name='Total deaths',
            hovertemplate="<b>Year %{x}</b><br>Total deaths: %{y:,}<extra></extra>",
        ),
        secondary_y=True
    )

    # Axes
    fig.update_yaxes(title_text="Percentage of deaths related to pollutants", range=[0, 100],
                     ticksuffix="%", secondary_y=False)
    fig.update_yaxes(title_text="Number of deaths", secondary_y=True)

    # Layout
    fig.update_layout(
        title="Yearly Comparison: % of Deaths Related to Pollutants (bars) vs Total Deaths (line)",
        bargap=0.15,
        hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(l=60, r=60, t=70, b=60)
    )
    
    # Add mean percentage line
    mean_percentage = yearly_comparison['percentage'].mean()
    fig.add_shape(
        type="line",
        x0=yearly_comparison['Year'].min(),
        x1=yearly_comparison['Year'].max(),
        y0=mean_percentage,
        y1=mean_percentage,
        line=dict(color="gray", width=2, dash="dash"),
        xref="x",
        yref="y"
    )
    
    # Add annotation for mean line
    fig.add_annotation(
        x=yearly_comparison['Year'].max(),
        y=mean_percentage,
        text=f"Mean: {mean_percentage:.2f}%",
        showarrow=True,
        arrowhead=1,
        ax=50,
        ay=-20,
        font=dict(size=12, color="gray")
    )

    return fig

fig = plot_pollutant_deaths_plotly(df, filtered_df)
fig.show()

# 7. Take Top-10* Causes of Deaths for each pollutants for predictive modeling
Here, a small constraint is applied that the correlation must be greater than 0.4 in order to be considered into top 10. Then the highest 10 correlations are taken

In [30]:
# Get top 10 correlations for each pollutant
hap_top10 = cause_corr_weighted[(cause_corr_weighted['Pollutant'] == 'HAP') & (cause_corr_weighted['r']> 0.4)].nlargest(10, 'r')['Cause'].tolist()
ozone_top10 = cause_corr_weighted[(cause_corr_weighted['Pollutant'] == 'OZONE') & (cause_corr_weighted['r']> 0.4)].nlargest(10, 'r')['Cause'].tolist()
no2_top10 = cause_corr_weighted[(cause_corr_weighted['Pollutant'] == 'NO2') & (cause_corr_weighted['r']> 0.4)].nlargest(10, 'r')['Cause'].tolist()
pm25_top10 = cause_corr_weighted[(cause_corr_weighted['Pollutant'] == 'PM25') & (cause_corr_weighted['r']> 0.4)].nlargest(10, 'r')['Cause'].tolist()

# Create a set of unique causes from all lists
unique_top_causes = list(set(hap_top10 + ozone_top10 + no2_top10 + pm25_top10))

# Print the results
print("Top 10 HAP correlations:")
print(hap_top10)
print("\nTop 10 OZONE correlations:")
print(ozone_top10)
print("\nTop 10 NO2 correlations:")
print(no2_top10)
print("\nTop 10 PM25 correlations:")
print(pm25_top10)
print("\nUnique causes across all pollutants:")
print(unique_top_causes)
print("Total Unique Causes: ", len(unique_top_causes))

Top 10 HAP correlations:
['Drug-susceptible tuberculosis', 'Varicella and herpes zoster', 'Neonatal encephalopathy due to birth asphyxia and trauma', 'Other neonatal disorders', 'Maternal hemorrhage', 'Diarrheal diseases', 'Pertussis', 'Typhoid fever', 'Drowning', 'Maternal obstructed labor and uterine rupture']

Top 10 OZONE correlations:
['Rheumatic heart disease', 'Chronic obstructive pulmonary disease']

Top 10 NO2 correlations:
['Tracheal, bronchus, and lung cancer', 'Total cancers', 'Colon and rectum cancer', 'Pancreatic cancer', 'Rheumatoid arthritis', 'Other non-Hodgkin lymphoma', 'Ovarian cancer', 'Brain and central nervous system cancer', "Parkinson's disease", "Alzheimer's disease and other dementias"]

Top 10 PM25 correlations:
['Rheumatic heart disease', 'Encephalitis', 'Venomous animal contact', 'Acute hepatitis A', 'Paratyphoid fever', 'Hemolytic disease and other neonatal jaundice', 'Typhoid fever', 'Asthma', 'Other nutritional deficiencies', 'Acute hepatitis E']

Uniqu

In [18]:
# Filter df to keep only rows where Cause is in unique_top_causes
top_causes_df = df[df['Cause'].isin(unique_top_causes)].copy()

# Display info about the filtered dataframe
print(f"Original dataframe shape: {df.shape}")
print(f"Filtered dataframe shape: {top_causes_df.shape}")
print(f"Percentage of rows retained: {top_causes_df.shape[0]/df.shape[0]*100:.2f}%")

# Show the first 5 rows of the filtered dataframe
pd.DataFrame(top_causes_df.columns)

Original dataframe shape: (1460844, 11)
Filtered dataframe shape: (189720, 11)
Percentage of rows retained: 12.99%


,0
0,Cause
1,Country
2,Year
3,Deaths
4,Region Name
5,Exposure Mean HAP
6,Exposure Mean NO2
7,Exposure Mean OZONE
8,Exposure Mean PM25
9,Population


In [19]:
# Rename exposure columns to simpler names
top_causes_df = top_causes_df.rename(columns={
    'Exposure Mean HAP': 'HAP',
    'Exposure Mean NO2': 'NO2',
    'Exposure Mean OZONE': 'OZONE',
    'Exposure Mean PM25': 'PM25'
})

# Display the first few rows with new column names
top_causes_df.head()

,Cause,Country,Year,Deaths,Region Name,HAP,NO2,OZONE,PM25,Population,Mortality Rate
3,Acute hepatitis A,Argentina,1990,22.093804,Southern Latin America,0.10,18.90,33.9,19.0,32755901.0,0.067450
6,Acute hepatitis E,Argentina,1990,0.698845,Southern Latin America,0.10,18.90,33.9,19.0,32755901.0,0.002133
20,Rheumatoid arthritis,Myanmar,1990,58.031486,Southeast Asia,0.96,5.14,35.3,41.6,39817251.0,0.145745
55,Drowning,Guyana,1990,66.605160,Caribbean,0.19,3.46,21.9,24.3,749894.0,8.881943
82,Drowning,Mauritania,1990,131.445606,Western Africa,0.85,5.26,33.4,78.6,1951878.0,6.734315


In [20]:
# Export the prepared dataframe to CSV
top_causes_df.to_csv("predictive_modeling_data_prepared.csv", index=False)
print("Data exported successfully to 'predictive_modeling_data_prepared.csv'")

Data exported successfully to 'predictive_modeling_data_prepared.csv'
